# Préparation du datatset

In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# 🔹 Load Dataset
data_1 = pd.read_csv("./data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
data_2 = pd.read_csv("./data/2023-02-12.csv")

# 🔹 Normalize column names (remove spaces and lowercase)
data_1.columns = data_1.columns.str.replace(" ", "").str.lower()
data_2.columns = data_2.columns.str.replace(" ", "").str.lower()

# 🔹 Find common columns
common_columns = list(set(data_1.columns) & set(data_2.columns))

# 🔹 Keep only common columns
data_1 = data_1[common_columns]
data_2 = data_2[common_columns]

# 🔹 Concatenate both datasets
concatenated_data = pd.concat([data_1, data_2], ignore_index=True)


# 🔹 Identify Non-Numeric Columns
non_numeric_columns = concatenated_data.select_dtypes(exclude=[np.number]).columns.tolist()
print("⚠️ Non-numeric columns:", non_numeric_columns)

# 🔹 Drop Non-Numeric Columns (Except 'label')
non_numeric_columns = [col for col in non_numeric_columns if col != 'label']
concatenated_data.drop(columns=non_numeric_columns, inplace=True)

# 🔹 Handle Inf and NaN
concatenated_data.replace([np.inf, -np.inf], np.nan, inplace=True)  # Convert inf to NaN
concatenated_data.fillna(concatenated_data.mean(numeric_only=True), inplace=True)  # Replace NaN with column mean

# 🔹 Ensure All Values Are Numeric
for col in concatenated_data.columns:
    if col != 'label':  # ✅ Don't convert the label column!
        concatenated_data[col] = pd.to_numeric(concatenated_data[col], errors='coerce')

# 🔹 Separate Features and Labels
features = [col for col in concatenated_data.columns if col != 'label']
X = concatenated_data[features]
y = concatenated_data['label']

# 🔹 Convert Labels into Binary (Attack vs. Benign)
y_binary = np.where(y == "BENIGN", 0, 1)  # 0 = BENIGN, 1 = Attack

# 🔹 Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 🔹 Split Data
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(X_scaled, y_binary, test_size=0.2, random_state=42)

print("✅ Data Preprocessing Completed Successfully!")


⚠️ Non-numeric columns: ['label']
✅ Data Preprocessing Completed Successfully!


In [30]:
print("Unique values in 'label' column:", concatenated_data['label'].unique())
print("Label distribution:\n", concatenated_data['label'].value_counts())


Unique values in 'label' column: ['BENIGN' 'DDoS' 'adbhoney' 'ddospot' 'log4pot' 'cowrie' 'ciscoasa'
 'redispot' 'elasticpot' 'mailoney']
Label distribution:
 label
DDoS          128027
BENIGN         97718
ddospot        72287
cowrie          2106
log4pot         1388
ciscoasa         195
adbhoney         180
elasticpot        63
mailoney          49
redispot          35
Name: count, dtype: int64


# Entrainement du SVM:

In [32]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# 🔹 Train SVM Model
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_model.fit(X_train_svm, y_train_svm)

# 🔹 Evaluate SVM
y_pred_svm = svm_model.predict(X_test_svm)
print("SVM Performance (Attack Detection):")
print(classification_report(y_test_svm, y_pred_svm))


SVM Performance (Attack Detection):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19448
           1       1.00      1.00      1.00     40962

    accuracy                           1.00     60410
   macro avg       1.00      1.00      1.00     60410
weighted avg       1.00      1.00      1.00     60410



# Auto encode + softmax pour la classification du type d'attaque:

In [35]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# 🔹 Extract Only Attack Data for Autoencoder
attack_data = concatenated_data[concatenated_data['label'] != "BENIGN"]
X_attack = attack_data[features]
y_attack = attack_data['label']

# 🔹 Normalize Attack Features
X_scaled_attack = scaler.transform(X_attack)

# 🔹 Encode Labels for Multi-Class Classification
label_encoder = LabelEncoder()
y_encoded_attack = label_encoder.fit_transform(y_attack)
y_categorical_attack = tf.keras.utils.to_categorical(y_encoded_attack)

# 🔹 Train-Test Split for Autoencoder
X_train_ae, X_test_ae, y_train_ae, y_test_ae = train_test_split(X_scaled_attack, y_categorical_attack, test_size=0.2, random_state=42)

# 🔹 Define Autoencoder Model
input_dim = X_train_ae.shape[1]
encoder = Sequential([
    Dense(64, activation='relu', input_shape=(input_dim,)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu')  # Bottleneck layer
])

decoder = Sequential([
    Dense(32, activation='relu', input_shape=(16,)),
    Dense(64, activation='relu'),
    Dense(input_dim, activation='sigmoid')  # Reconstruction output
])

autoencoder = Sequential([encoder, decoder])

# 🔹 Train Autoencoder on Attack Data
autoencoder.compile(optimizer=Adam(0.001), loss='mse')
autoencoder.fit(X_train_ae, X_train_ae, epochs=10, batch_size=32, validation_data=(X_test_ae, X_test_ae))

# 🔹 Define Softmax Classifier on Bottleneck
classifier = Sequential([
    encoder,  # Use encoder from Autoencoder
    Dense(len(label_encoder.classes_), activation='softmax')  # Attack Type Classification
])

# 🔹 Train Softmax Classifier
classifier.compile(optimizer=Adam(0.0005), loss='categorical_crossentropy', metrics=['accuracy'])
classifier.fit(X_train_ae, y_train_ae, epochs=10, batch_size=32, validation_data=(X_test_ae, y_test_ae))

# 🔹 Evaluate Autoencoder + Softmax
loss, accuracy = classifier.evaluate(X_test_ae, y_test_ae)
print(f"Autoencoder + Softmax Performance - Accuracy: {accuracy:.4f}")


Epoch 1/10


c:\Users\ibrah\Desktop\IDIA-5A\PRED\PRED\venv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5109/5109 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.4547 - val_loss: 0.4835
Epoch 2/10
5109/5109 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.4791 - val_loss: 0.4783
Epoch 3/10
5109/5109 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.4536 - val_loss: 0.4775
Epoch 4/10
5109/5109 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.4373 - val_loss: 0.4765
Epoch 5/10
5109/5109 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.5182 - val_loss: 0.4764
Epoch 6/10
5109/5109 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.4649 - val_loss: 0.4764
Epoch 7/10
5109/5109 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.4204 - val_loss: 0.4766
Epoch 8/10
5109/5109 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.4800 - val_loss: 0.4763
Epoch 9/10
5109/5109 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 0.4354 - val_loss: 0.4763
Epoch 10/10
5109/5109 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.4755 - val_loss: 0.4762
Epoch 1/10
5109/5109 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9596 - loss: 0.2124 - val_accuracy: 0.9924 - val_lo

# Fusion des deux algorithmes:

In [36]:
# 🔹 Step 1: Predict Attack/Normal with SVM
svm_predictions = svm_model.predict(X_test_svm)

# 🔹 Step 2: Select Only Attack Samples for Autoencoder
attack_indices = np.where(svm_predictions == 1)[0]
X_test_attacks = X_test_svm[attack_indices]  # Only attack samples

# 🔹 Step 3: Predict Attack Type with Autoencoder
autoencoder_predictions = classifier.predict(X_test_attacks)
attack_types = np.argmax(autoencoder_predictions, axis=1)  # Get attack type indices

# 🔹 Step 4: Convert Attack Type Indices to Labels
attack_labels = label_encoder.inverse_transform(attack_types)

# 🔹 Step 5: Merge Results
final_predictions = np.array(["BENIGN"] * len(X_test_svm))  # Start with "BENIGN"
final_predictions[attack_indices] = attack_labels  # Replace attacks with classified types

# 🔹 Step 6: Evaluate Hybrid Model
y_test_labels = concatenated_data.iloc[pd.DataFrame(X_test_svm, columns=features).index]['label'].values
print("Hybrid Model Performance (SVM + Autoencoder):")
print(classification_report(y_test_labels, final_predictions))


1281/1281 ━━━━━━━━━━━━━━━━━━━━ 1s 982us/step
Hybrid Model Performance (SVM + Autoencoder):
              precision    recall  f1-score   support

      BENIGN       0.50      0.32      0.39     30012
        DDoS       0.50      0.43      0.46     30398
      adbhon       0.00      0.00      0.00         0
      ciscoa       0.00      0.00      0.00         0
      cowrie       0.00      0.00      0.00         0
      ddospo       0.00      0.00      0.00         0
      log4po       0.00      0.00      0.00         0
      redisp       0.00      0.00      0.00         0

    accuracy                           0.37     60410
   macro avg       0.13      0.09      0.11     60410
weighted avg       0.50      0.37      0.43     60410



c:\Users\ibrah\Desktop\IDIA-5A\PRED\PRED\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ibrah\Desktop\IDIA-5A\PRED\PRED\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\ibrah\Desktop\IDIA-5A\PRED\PRED\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
